In [1]:
from fifa.matches import wc_matches, check_match_counts, latest_year
from fifa.ingest import load_rankings, load_rankings_json
import pandas as pd
from fifa import config
import openpyxl

In [2]:
matches = wc_matches()
rankings= load_rankings()

In [3]:

rankings = pd.concat(
    [
        load_rankings(),                                  # CSV, through 2018
        load_rankings_json(config.FIFA_RANKING_JSON_2022, "2022-10-06"),      # 2022
        load_rankings_json(config.FIFA_RANKING_JSON_2026, "2026-06-05"),      # 2026
    ],
    ignore_index=True,
)

In [4]:
# Earliest match date per tournament
wc_start_date = matches.groupby("year")["date"].min()
wc_years = list(wc_start_date.index)
wc_start_date

year
1994   1994-06-17
1998   1998-06-10
2002   2002-05-31
2006   2006-06-09
2010   2010-06-11
2014   2014-06-12
2018   2018-06-14
2022   2022-11-20
2026   2026-06-11
Name: date, dtype: datetime64[us]

In [5]:
# Tag each ranking row with its year, then list the snapshot dates available per year.
rankings["year"] = rankings["rank_date"].dt.year
snapshot_dates = rankings.groupby("year")["rank_date"].unique()

# For each WC year, the latest snapshot on or before kickoff
closest_rank_date = {}
for year in wc_years:
    if year not in snapshot_dates:
        continue
    eligible = [d for d in snapshot_dates[year] if d <= wc_start_date[year]]
    if eligible:
        closest_rank_date[year] = max(eligible)
closest_rank_date

{1994: Timestamp('1994-06-14 00:00:00'),
 1998: Timestamp('1998-05-20 00:00:00'),
 2002: Timestamp('2002-05-15 00:00:00'),
 2006: Timestamp('2006-05-17 00:00:00'),
 2010: Timestamp('2010-05-26 00:00:00'),
 2014: Timestamp('2014-06-05 00:00:00'),
 2018: Timestamp('2018-06-07 00:00:00'),
 2022: Timestamp('2022-10-06 00:00:00'),
 2026: Timestamp('2026-06-05 00:00:00')}

In [6]:
# Map each year to its chosen snapshot date
rankings["snapshot_date"] = rankings["year"].map(closest_rank_date)

# Keep only the snapshot rows -> one rank per (year, country).
rank_lookup = rankings[rankings["rank_date"] == rankings["snapshot_date"]][
    ["year", "country_full", "rank"]
].copy()

In [7]:
rank_lookup[rank_lookup["country_full"].str.contains("Türkiye")]

,year,country_full,rank
57837,2022,Türkiye,45
58025,2026,Türkiye,22


In [8]:
# FIFA ranking name -> results-dataset name.
name_fixes_rankings = {
    "Korea Republic": "South Korea",
    "Korea DPR": "North Korea",
    "China PR": "China",
    "USA": "United States",
    "IR Iran": "Iran",
    "Czechia": "Czech Republic",
    "Congo DR": "DR Congo",
    "Türkiye": "Turkey",
    "Cabo Verde": "Cape Verde",
    "Cape Verde Islands": "Cape Verde",
    "Côte d'Ivoire": "Ivory Coast",
    "Serbia and Montenegro":"Serbia"

}
rank_lookup["country_full"] = rank_lookup["country_full"].replace(name_fixes_rankings)

In [9]:
# Left-merge the snapshot rank onto team_a, then team_b.
matches = matches.merge(
    rank_lookup.rename(columns={"country_full": "team_a", "rank": "team_a_rank"}),
    on=["year", "team_a"],
    how="left",
)
matches = matches.merge(
    rank_lookup.rename(columns={"country_full": "team_b", "rank": "team_b_rank"}),
    on=["year", "team_b"],
    how="left",
)

In [10]:
# Drop all entries whose rank is not available - at this point that's only Iran(1998) & Serbia(1998)
matches = matches[~(matches["team_a_rank"].isna() | matches["team_b_rank"].isna())]
matches

,date,year,stage,team_a,team_b,team_a_score,team_b_score,outcome,winner,decided_by_shootout,is_host_match,city,country,team_a_rank,team_b_rank
0,1994-06-17,1994,group,Germany,Bolivia,1,0,team_a_win,Germany,False,False,Chicago,United States,1,43
1,1994-06-17,1994,group,Spain,South Korea,2,2,draw,<NA>,False,False,Dallas,United States,5,37
2,1994-06-18,1994,group,Colombia,Romania,1,3,team_b_win,Romania,False,False,Pasadena,United States,17,7
3,1994-06-18,1994,group,Italy,Republic of Ireland,0,1,team_b_win,Republic of Ireland,False,False,East Rutherford,United States,4,14
4,1994-06-18,1994,group,United States,Switzerland,1,1,draw,<NA>,False,True,Pontiac,United States,23,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599,2026-07-11,2026,knockout,Norway,England,1,2,team_b_win,England,False,False,Miami Gardens,United States,31,4
600,2026-07-14,2026,knockout,France,Spain,0,2,team_b_win,Spain,False,False,Arlington,United States,3,2
601,2026-07-15,2026,knockout,England,Argentina,1,2,team_b_win,Argentina,False,False,Atlanta,United States,4,1
602,2026-07-18,2026,knockout,France,England,4,6,team_b_win,England,False,False,Miami Gardens,United States,3,4


In [11]:
matches.to_excel("rankings_and_matches_wc.xlsx")

In [12]:
matches["rank_diff"] = matches['team_a_rank'] - matches['team_b_rank']


In [ ]:
matches["high_ranked_won"] = (((matches['team_a_rank'] < matches['team_b_rank']) & (matches['outcome'] == 'team_a_win')) | ((matches['team_a_rank'] > matches['team_b_rank']) & (matches['outcome'] == 'team_b_win')) )

,date,year,stage,team_a,team_b,team_a_score,team_b_score,outcome,winner,decided_by_shootout,is_host_match,city,country,team_a_rank,team_b_rank,rank_diff,high_ranked_won
1,1994-06-17,1994,group,Spain,South Korea,2,2,draw,<NA>,False,False,Dallas,United States,5,37,-32,False
4,1994-06-18,1994,group,United States,Switzerland,1,1,draw,<NA>,False,True,Pontiac,United States,23,12,11,False
6,1994-06-19,1994,group,Cameroon,Sweden,2,2,draw,<NA>,False,False,Pasadena,United States,24,10,14,False
11,1994-06-21,1994,group,Germany,Spain,1,1,draw,<NA>,False,False,Chicago,United States,1,5,-4,False
16,1994-06-23,1994,group,South Korea,Bolivia,0,0,draw,<NA>,False,False,Foxborough,United States,37,43,-6,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557,2026-06-25,2026,group,Paraguay,Australia,0,0,draw,<NA>,False,False,Santa Clara,United States,41,27,14,False
560,2026-06-26,2026,group,Cape Verde,Saudi Arabia,0,0,draw,<NA>,False,False,Houston,United States,67,61,6,False
561,2026-06-26,2026,group,Egypt,Iran,1,1,draw,<NA>,False,False,Seattle,United States,29,20,9,False
566,2026-06-27,2026,group,Algeria,Austria,3,3,draw,<NA>,False,False,Kansas City,United States,28,24,4,False


In [14]:
# === Statistics for visualization ===
# Derived column + a Wilson confidence-interval helper (robust for small n).
import numpy as np

# Absolute ranking gap between the two teams in a match.
matches["rank_gap"] = matches["rank_diff"].abs()

def wilson_ci(wins, n, z=1.96):
    # 95% Wilson score interval for a proportion; better than normal approx for small n.
    if n == 0:
        return (np.nan, np.nan)
    p = wins / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (centre - half, centre + half)

In [15]:
# 1) Overall: baseline win rate vs. the draws-as-loss ceiling.
n = len(matches)
wins = int(matches["high_ranked_won"].sum())
draws = int((matches["outcome"] == "draw").sum())
win_rate = wins / n
draw_rate = draws / n
ceiling = 1 - draw_rate

overall_stats = pd.DataFrame({
    "Metric": [
        "Matches analysed (with ranks)",
        "Higher-ranked team wins",
        "Higher-ranked win rate (%)",
        "Draws",
        "Draw rate (%)",
        "Strategy ceiling = 1 - draw rate (%)",
        "Share of ceiling achieved (%)",
    ],
    "Value": [
        n,
        wins,
        round(win_rate * 100, 1),
        draws,
        round(draw_rate * 100, 1),
        round(ceiling * 100, 1),
        round(win_rate / ceiling * 100, 1),
    ],
})
overall_stats

,Metric,Value
0,Matches analysed (with ranks),600.0
1,Higher-ranked team wins,350.0
2,Higher-ranked win rate (%),58.3
3,Draws,110.0
4,Draw rate (%),18.3
5,Strategy ceiling = 1 - draw rate (%),81.7
6,Share of ceiling achieved (%),71.4


In [16]:
# 2) By tournament: higher-ranked win rate and draw rate per World Cup.
by_tournament = (
    matches.assign(is_draw=matches["outcome"].eq("draw"))
    .groupby("year")
    .agg(
        matches=("high_ranked_won", "size"),
        higher_ranked_wins=("high_ranked_won", "sum"),
        draws=("is_draw", "sum"),
    )
    .reset_index()
)
by_tournament["win_rate_pct"] = (by_tournament["higher_ranked_wins"] / by_tournament["matches"] * 100).round(1)
by_tournament["draw_rate_pct"] = (by_tournament["draws"] / by_tournament["matches"] * 100).round(1)
by_tournament = by_tournament.rename(columns={
    "year": "Year",
    "matches": "Matches",
    "higher_ranked_wins": "Higher-ranked wins",
    "draws": "Draws",
    "win_rate_pct": "Higher-ranked win rate (%)",
    "draw_rate_pct": "Draw rate (%)",
})
by_tournament

,Year,Matches,Higher-ranked wins,Draws,Higher-ranked win rate (%),Draw rate (%)
0,1994,52,31,8,59.6,15.4
1,1998,60,30,15,50.0,25.0
2,2002,64,34,14,53.1,21.9
3,2006,64,32,11,50.0,17.2
4,2010,64,37,14,57.8,21.9
5,2014,64,44,9,68.8,14.1
6,2018,64,34,9,53.1,14.1
7,2022,64,39,10,60.9,15.6
8,2026,104,69,20,66.3,19.2


In [17]:
# 3) By stage: group vs. knockout.
by_stage = (
    matches.groupby("stage")
    .agg(matches=("high_ranked_won", "size"), higher_ranked_wins=("high_ranked_won", "sum"))
    .reset_index()
)
by_stage["win_rate_pct"] = (by_stage["higher_ranked_wins"] / by_stage["matches"] * 100).round(1)
by_stage = by_stage.rename(columns={
    "stage": "Stage",
    "matches": "Matches",
    "higher_ranked_wins": "Higher-ranked wins",
    "win_rate_pct": "Higher-ranked win rate (%)",
})
by_stage

,Stage,Matches,Higher-ranked wins,Higher-ranked win rate (%)
0,group,441,237,53.7
1,knockout,159,113,71.1


In [18]:
# 4) By ranking gap: win rate per gap bucket with 95% Wilson CIs.
gap_bins = [0, 3, 6, 10, 15, 20, 30, 50, 100, 300]
matches["rank_gap_bucket"] = pd.cut(matches["rank_gap"], bins=gap_bins, include_lowest=True)
by_gap = (
    matches.groupby("rank_gap_bucket", observed=True)
    .agg(matches=("high_ranked_won", "size"), higher_ranked_wins=("high_ranked_won", "sum"))
    .reset_index()
)
_ci = by_gap.apply(lambda r: wilson_ci(float(r["higher_ranked_wins"]), float(r["matches"])), axis=1)
by_gap["win_rate_pct"] = (by_gap["higher_ranked_wins"] / by_gap["matches"] * 100).round(1)
by_gap["ci_low_pct"] = (_ci.str[0] * 100).round(1)
by_gap["ci_high_pct"] = (_ci.str[1] * 100).round(1)
by_gap["rank_gap_bucket"] = by_gap["rank_gap_bucket"].astype(str)
by_gap = by_gap.rename(columns={
    "rank_gap_bucket": "Rank gap bucket",
    "matches": "Matches",
    "higher_ranked_wins": "Higher-ranked wins",
    "win_rate_pct": "Win rate (%)",
    "ci_low_pct": "95% CI low (%)",
    "ci_high_pct": "95% CI high (%)",
})
by_gap

,Rank gap bucket,Matches,Higher-ranked wins,Win rate (%),95% CI low (%),95% CI high (%)
0,"(-0.001, 3.0]",68,36,52.9,41.2,64.3
1,"(3.0, 6.0]",65,31,47.7,36.0,59.6
2,"(6.0, 10.0]",82,44,53.7,42.9,64.0
3,"(10.0, 15.0]",73,37,50.7,39.5,61.8
4,"(15.0, 20.0]",72,40,55.6,44.1,66.5
5,"(20.0, 30.0]",100,69,69.0,59.4,77.2
6,"(30.0, 50.0]",104,69,66.3,56.8,74.7
7,"(50.0, 100.0]",34,22,64.7,47.9,78.5
8,"(100.0, 300.0]",2,2,100.0,34.2,100.0


In [19]:
# 5) Threshold sweep: bet the favourite only when the gap exceeds X, else coin-flip (50%).
N = len(matches)
sweep_rows = []
for X in range(0, int(matches["rank_gap"].max()) + 1):
    bet = matches["rank_gap"] > X
    n_bet = int(bet.sum())
    n_flip = N - n_bet
    expected_wins = float(matches.loc[bet, "high_ranked_won"].sum()) + 0.5 * n_flip
    sweep_rows.append({
        "Gap threshold X": X,
        "Matches bet": n_bet,
        "Matches coin-flipped": n_flip,
        "Expected win rate (%)": round(expected_wins / N * 100, 1),
    })
threshold_sweep = pd.DataFrame(sweep_rows)
best_X = threshold_sweep.loc[threshold_sweep["Expected win rate (%)"].idxmax()]
print("Best threshold X =", int(best_X["Gap threshold X"]), "-> expected win rate", best_X["Expected win rate (%)"], "%")
threshold_sweep.head()

Best threshold X = 0 -> expected win rate 58.3 %


,Gap threshold X,Matches bet,Matches coin-flipped,Expected win rate (%)
0,0,600,0,58.3
1,1,570,30,58.3
2,2,552,48,58.0
3,3,532,68,58.0
4,4,514,86,58.0


In [20]:
# 6) Host effect: higher-ranked win rate for host vs. non-host matches.
host_effect = (
    matches.groupby("is_host_match")
    .agg(matches=("high_ranked_won", "size"), higher_ranked_wins=("high_ranked_won", "sum"))
    .reset_index()
)
host_effect["win_rate_pct"] = (host_effect["higher_ranked_wins"] / host_effect["matches"] * 100).round(1)
host_effect["is_host_match"] = host_effect["is_host_match"].map({True: "Host match", False: "Non-host match"})
host_effect = host_effect.rename(columns={
    "is_host_match": "Match type",
    "matches": "Matches",
    "higher_ranked_wins": "Higher-ranked wins",
    "win_rate_pct": "Higher-ranked win rate (%)",
})
host_effect

,Match type,Matches,Higher-ranked wins,Higher-ranked win rate (%)
0,Non-host match,538,312,58.0
1,Host match,62,38,61.3


In [21]:
# 7) Biggest upsets: lower-ranked team won, largest ranking gaps first.
_ups = matches[(~matches["high_ranked_won"]) & (matches["outcome"] != "draw")].copy()
_ups["Winner rank"] = np.where(_ups["winner"] == _ups["team_a"], _ups["team_a_rank"], _ups["team_b_rank"])
_ups["Loser"] = np.where(_ups["winner"] == _ups["team_a"], _ups["team_b"], _ups["team_a"])
_ups["Loser rank"] = np.where(_ups["winner"] == _ups["team_a"], _ups["team_b_rank"], _ups["team_a_rank"])
biggest_upsets = (
    _ups.sort_values("rank_gap", ascending=False)
    .head(20)[["year", "stage", "winner", "Winner rank", "Loser", "Loser rank", "rank_gap"]]
    .rename(columns={"year": "Year", "stage": "Stage", "winner": "Winner", "rank_gap": "Rank gap"})
    .reset_index(drop=True)
)
biggest_upsets

,Year,Stage,Winner,Winner rank,Loser,Loser rank,Rank gap
0,2010,group,South Africa,83,France,9,74
1,2018,knockout,Russia,70,Spain,10,60
2,1998,group,Nigeria,74,Spain,15,59
3,2018,group,South Korea,57,Germany,1,56
4,2022,group,Saudi Arabia,51,Argentina,3,48
5,2006,group,Ghana,48,Czech Republic,2,46
6,2018,group,Japan,61,Colombia,16,45
7,2006,group,Ghana,48,United States,5,43
8,2022,group,Cameroon,43,Brazil,1,42
9,2002,group,Senegal,42,France,1,41


In [22]:
# 8) Semi-finalist alignment: did the pre-tournament top-4 ranked teams reach the semis?
# Per-participant rank each year (stack both sides, dedupe).
_side_a = matches[["year", "team_a", "team_a_rank"]].rename(columns={"team_a": "team", "team_a_rank": "rank"})
_side_b = matches[["year", "team_b", "team_b_rank"]].rename(columns={"team_b": "team", "team_b_rank": "rank"})
team_ranks = pd.concat([_side_a, _side_b], ignore_index=True).drop_duplicates(["year", "team"])

sf_rows = []
for year, g in matches.groupby("year"):
    # Semi-finalists = the four teams in the last two matches (3rd-place playoff + final).
    last_two = g.sort_values("date").tail(2)
    semifinalists = set(last_two["team_a"]).union(last_two["team_b"])
    top4 = team_ranks[team_ranks["year"] == year].nsmallest(4, "rank")
    top4_teams = list(top4.sort_values("rank")["team"])
    overlap = semifinalists.intersection(top4_teams)
    sf_rows.append({
        "Year": int(year),
        "Top-4 ranked (entering)": ", ".join(top4_teams),
        "Semi-finalists": ", ".join(sorted(semifinalists)),
        "Top-4 who reached semis": len(overlap),
        "All four reached semis": len(overlap) == 4,
    })
semifinalist_alignment = pd.DataFrame(sf_rows)
semifinalist_alignment

,Year,Top-4 ranked (entering),Semi-finalists,Top-4 who reached semis,All four reached semis
0,1994,"Germany, Netherlands, Brazil, Italy","Brazil, Bulgaria, Italy, Sweden",2,False
1,1998,"Brazil, Germany, Mexico, England","Brazil, Croatia, France, Netherlands",1,False
2,2002,"France, Brazil, Argentina, Portugal","Brazil, Germany, South Korea, Turkey",1,False
3,2006,"Brazil, Czech Republic, Netherlands, Mexico","France, Germany, Italy, Portugal",0,False
4,2010,"Brazil, Spain, Portugal, Netherlands","Germany, Netherlands, Spain, Uruguay",2,False
5,2014,"Spain, Germany, Brazil, Portugal","Argentina, Brazil, Germany, Netherlands",2,False
6,2018,"Germany, Brazil, Belgium, Portugal","Belgium, Croatia, England, France",1,False
7,2022,"Brazil, Belgium, Argentina, France","Argentina, Croatia, France, Morocco",2,False
8,2026,"Argentina, Spain, France, England","Argentina, England, France, Spain",4,True


In [23]:
# 9) Export every stat table to a cleanly labelled Excel workbook (one sheet each).
from openpyxl.utils import get_column_letter

definitions = pd.DataFrame({
    "Sheet": [
        "Overall", "By tournament", "By stage", "By rank gap",
        "Threshold sweep", "Host effect", "Biggest upsets", "Semifinalist alignment",
    ],
    "What it shows": [
        "Baseline: how often the higher-ranked team won, vs. the ceiling set by draws.",
        "Higher-ranked win rate and draw rate for each World Cup.",
        "Win rate split by group stage vs. knockout stage.",
        "Win rate by size of the ranking gap, with 95% confidence intervals.",
        "Win rate if you only bet when the gap exceeds X (coin-flip below X).",
        "Higher-ranked win rate for host vs. non-host matches.",
        "Matches where the lower-ranked team won, largest ranking gaps first.",
        "Whether each tournament's four highest-ranked teams reached the semi-finals.",
    ],
})
notes = pd.DataFrame({
    "Note": [
        "Higher-ranked = lower FIFA rank number.",
        "'draws = loss': a draw counts against the higher-ranked team, so the max possible win rate is 1 - draw rate (the 'ceiling').",
        "Knockout ties decided by penalties count as a win for the team that advanced.",
        "Scope: World Cup matches 1994-2026 where both teams have a pre-tournament rank (2 unranked 1998 matches dropped).",
        "Ranking snapshot = last FIFA release on/before each tournament's kickoff.",
        "Semi-finalists derived as the four teams in the last two matches (3rd-place playoff + final).",
    ]
})

sheets = {
    "Definitions": definitions,
    "Notes": notes,
    "Overall": overall_stats,
    "By tournament": by_tournament,
    "By stage": by_stage,
    "By rank gap": by_gap,
    "Threshold sweep": threshold_sweep,
    "Host effect": host_effect,
    "Biggest upsets": biggest_upsets,
    "Semifinalist alignment": semifinalist_alignment,
}

with pd.ExcelWriter("wc_ranking_stats.xlsx", engine="openpyxl") as writer:
    for name, df in sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        for i, col in enumerate(df.columns, start=1):
            lengths = df[col].astype(str).map(len)
            width = max(int(lengths.max()) if len(lengths) else 0, len(str(col))) + 2
            ws.column_dimensions[get_column_letter(i)].width = min(width, 60)

print("Wrote wc_ranking_stats.xlsx with", len(sheets), "sheets")

Wrote wc_ranking_stats.xlsx with 10 sheets
